# Notebook 2: teacher -> YOLOv8-seg knowledge distillation

**Goal:** train a YOLOv8 segmentation model light enough for a **Raspberry Pi** in the coop, using a large text-prompted model as the **teacher**, so no masks have to be drawn by hand. The teacher is **Grounded SAM 2** (Grounding DINO + SAM 2.1, public) by default, or **SAM 3** once Meta approves your access. Switch with `TEACHER` in the Settings cell.

```
folder of raw videos -> sample frames -> teacher masks (broiler / scale / feeder)  [teacher]
   -> YOLO-seg polygons, split by video -> train yolov8n-seg                      [student]
   -> compare student vs teacher (IoU, counts, speed) -> export NCNN for the Pi
   -> weight estimation with student masks vs teacher masks on FlockLabel data
```

**Install once:**

```bash
pip install -e ".[all]"      # teachers + ultralytics + notebook extras
# only for TEACHER = "sam3": request access at huggingface.co/facebook/sam3, then log in
```

Training runs on CUDA, Apple **MPS** or CPU, and the device is picked automatically. Labelling with the teacher is the slow step, so start with ~10-20 frames per video.

> Set `DRY_RUN = True` to walk through the notebook on synthetic videos first. In that mode, training runs only if `ultralytics` is installed. Otherwise a stand-in student is used so every plot still appears.

## 0. Settings

In [ ]:
# ---- change these -------------------------------------------------------------
DRY_RUN = False
TEACHER = "grounded_sam2"        # "grounded_sam2" (public) or "sam3" (once Meta approves access)
FRAMES_PER_VIDEO = 15            # frames the teacher labels per video
SAMPLING = "uniform"             # "uniform", "random" or "motion" (busiest frames)
SKIP_START = 0                   # ignore the first N frames of each video
VAL_FRACTION = 0.2               # share of VIDEOS kept for validation
MIN_TEACHER_SCORE = 0.4          # drop low-confidence teacher masks from the training labels
BASE_MODEL = "yolov8n-seg.pt"    # nano = Raspberry-Pi size; "yolov8s-seg.pt" if accuracy is short
EPOCHS = 100
IMGSZ = 640                      # 320 or 480 run faster on the Pi; retrain at that size
BATCH = 16
RUN_TRAINING = True
EXPORT_FORMATS = ("ncnn", "onnx")
SEED = 0
SAVE_FIGURES = True
OUTPUT_DIR = "~/broilerseg_outputs/02_distillation"   # any writable folder; ~ = your home folder
# -------------------------------------------------------------------------------

In [ ]:
import warnings
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

import broilerseg
from broilerseg import viz
from broilerseg.config import CLASS_NAMES, PromptConfig, VolumeConfig
from broilerseg.distill import (benchmark, export_for_raspberry_pi, load_training_curves,
                                path_size_mb, teacher_student_table, train_student)
from broilerseg.evaluation import event_table, regression_metrics
from broilerseg.experiments import WeightExperiment
from broilerseg.io import (load_flocklabel_export, motion_scores, read_frame, scan_videos,
                           subsample_per_event)
from broilerseg.labeling import auto_label_videos, load_result, plan_sampling
from broilerseg.notebook import ask_for_directory, environment_report
from broilerseg.yolo_dataset import build_yolo_dataset

warnings.filterwarnings("ignore", category=UserWarning)
viz.set_style()
OUT = Path(OUTPUT_DIR).expanduser().resolve(); (OUT / "figures").mkdir(parents=True, exist_ok=True)
FIG = 0
def show(fig, name):
    global FIG
    FIG += 1
    if SAVE_FIGURES:
        viz.save(fig, OUT / "figures" / f"{FIG:02d}_{name}.png")
    plt.show()

try:
    import ultralytics  # noqa: F401
    HAVE_ULTRALYTICS = True
except ImportError:
    HAVE_ULTRALYTICS = False
print("broilerseg", broilerseg.__version__, "| ultralytics available:", HAVE_ULTRALYTICS)
display(pd.Series(environment_report(), name="environment").to_frame())

## 1. Point the notebook at the folder of raw videos (mask-creation data)
Every video under the folder, including subfolders, is scanned. To skip the prompt next time, set `BROILERSEG_VIDEO_DIR`.

In [ ]:
if DRY_RUN:
    from broilerseg.synthetic import make_demo_data
    demo = make_demo_data(OUT / "demo_data", n_events=8, frames_per_event=30)
    VIDEO_DIR = demo["videos"]
    print("DRY RUN: synthetic videos in", VIDEO_DIR)
else:
    VIDEO_DIR = ask_for_directory("Paste the folder of videos to label with the teacher",
                                  env_var="BROILERSEG_VIDEO_DIR")
videos = scan_videos(VIDEO_DIR)
display(videos.drop(columns="path"))
show(viz.plot_video_inventory(videos), "video_inventory")
first = [read_frame(p, min(10, max(0, n - 1))) for p, n in zip(videos.path[:15], videos.n_frames[:15])]
show(viz.plot_contact_sheet(first, list(videos.stem[:15]), title="One frame from each video"), "contact_sheet")

## 2. Choose which frames to label
`uniform` spreads frames evenly through each video. `motion` picks the busiest moments, such as birds stepping on and off the scale. It needs one pass over each video, so start with `uniform` on a large folder.

In [ ]:
plan = plan_sampling(videos, FRAMES_PER_VIDEO, SAMPLING, SEED, SKIP_START)
plan.to_csv(OUT / "sampling_plan.csv", index=False)
show(viz.plot_sampling_plan(plan), "sampling_plan")
v0 = videos.iloc[0]
mo = motion_scores(v0.path, stride=max(1, int(v0.n_frames) // 300))
show(viz.plot_motion_profile(mo, plan[plan.video == v0.path].frame_idx.tolist(),
                             title=f"Activity in {v0.stem}"), "motion_profile")

## 3. Label the frames with the teacher
Frames, masks (`.npz`), quick-look overlays and `manifest.csv` go to `labels/`. **Re-running resumes**: frames that already have masks are skipped.

In [ ]:
LABEL_DIR = OUT / "labels"
if DRY_RUN:
    from broilerseg.synthetic import ColorSegmenter
    teacher = ColorSegmenter()
else:
    from broilerseg.segmentation import load_teacher
    # Grounded SAM 2 options: dino="tiny"|"base", sam2="tiny"|"small"|"base_plus"|"large"
    teacher = load_teacher(TEACHER, prompts=PromptConfig(), max_side=1280)
    print(teacher.name, "on", teacher.device)
manifest = auto_label_videos(VIDEO_DIR, LABEL_DIR, teacher, plan=plan)
print(f"{len(manifest)} frames labelled -> {LABEL_DIR.resolve()}")
display(manifest.head())

### 3a. What did the teacher produce?

In [ ]:
show(viz.plot_label_statistics(manifest), "label_statistics")
def load_frame(stem):
    return cv2.cvtColor(cv2.imread(str(LABEL_DIR / "frames" / f"{stem}.jpg")), cv2.COLOR_BGR2RGB)
rng = np.random.default_rng(SEED)
stems = rng.choice(manifest.stem, size=min(12, len(manifest)), replace=False)
teacher_results = {st: load_result(LABEL_DIR / "masks" / f"{st}.npz") for st in manifest.stem}
show(viz.plot_gallery([(load_frame(st), teacher_results[st], st[-22:]) for st in stems], ncols=4,
                      title=f"{teacher.name} pseudo-labels on random frames"), "teacher_gallery")
show(viz.plot_score_distributions([teacher_results[s] for s in manifest.stem]), "teacher_scores")
st = stems[0]
show(viz.plot_segmentation_panels(load_frame(st), teacher_results[st], st), "teacher_panels")
show(viz.plot_instances_by_prompt(load_frame(st), teacher_results[st]), "teacher_prompts")

### 3b. Frames the teacher may have got wrong
Frames with **no feeder**, **no scale** or **several feeders** are worth checking by eye before training. Wrong pseudo-labels become the student's mistakes.

In [ ]:
suspect = manifest[(manifest.n_feeder != 1) | (manifest.n_scale != 1)]
print(f"{len(suspect)} suspicious frames out of {len(manifest)}")
display(suspect[["stem", "n_broiler", "n_scale", "n_feeder", "score_feeder", "score_scale"]].head(20))
if len(suspect):
    show(viz.plot_gallery([(load_frame(s), teacher_results[s], s[-22:]) for s in suspect.stem[:8]],
                          ncols=4, show_scores=True, title="Check these pseudo-labels"), "teacher_suspect")

## 4. Build the YOLO-seg dataset
Every mask becomes a polygon. **Train/val are split by video**, so near-duplicate frames don't leak between them. *Polygon fidelity* is the IoU between a teacher mask and its polygon, and shows how much the conversion loses.

In [ ]:
YOLO_DIR = OUT / "yolo_dataset"
ds = build_yolo_dataset(LABEL_DIR, YOLO_DIR, (1 - VAL_FRACTION, VAL_FRACTION, 0.0), seed=SEED,
                        min_score=MIN_TEACHER_SCORE)
print((YOLO_DIR / "data.yaml").read_text())
display(ds.groupby("split")[[f"n_{c}" for c in CLASS_NAMES]].sum())
show(viz.plot_split_summary(ds), "split_summary")
fig, axes = plt.subplots(1, 4, figsize=(22, 4.5))
for ax, (_, r) in zip(axes, ds.sample(min(4, len(ds)), random_state=SEED).iterrows()):
    viz.plot_yolo_label_check(YOLO_DIR / "images" / r.split / f"{r.stem}.jpg",
                              YOLO_DIR / "labels" / r.split / f"{r.stem}.txt", ax=ax, title=f"{r.split}: {r.stem[-18:]}")
fig.suptitle("Label files read back from disk (what YOLO will train on)", fontweight="bold"); fig.tight_layout()
show(fig, "yolo_label_check")

## 5. Train the student (YOLOv8n-seg)
The student starts from COCO-pretrained weights and is fine-tuned on the teacher's labels. Ultralytics writes its own plots (confusion matrix, PR curves, validation predictions) into the run folder, and they're shown below.

In [ ]:
RUN_DIR = None
if RUN_TRAINING and HAVE_ULTRALYTICS:
    RUN_DIR = train_student(YOLO_DIR / "data.yaml", BASE_MODEL, EPOCHS, IMGSZ, BATCH,
                            project=OUT / "runs", name="student", seed=SEED)
    print("Run folder:", RUN_DIR)
elif not HAVE_ULTRALYTICS:
    print("ultralytics is not installed: pip install -e '.[yolo]'  (skipping training)")

In [ ]:
if RUN_DIR is not None:
    curves = load_training_curves(RUN_DIR)
    display(curves.tail())
    show(viz.plot_training_curves(curves), "training_curves")
    pngs = [p for p in sorted(Path(RUN_DIR).glob("*.png")) + sorted(Path(RUN_DIR).glob("*.jpg"))
            if not p.name.startswith("train_batch")][:12]
    if pngs:
        show(viz.plot_contact_sheet([cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB) for p in pngs],
                                    [p.name for p in pngs], ncols=3, size=5.5,
                                    title="Ultralytics plots from the run folder"), "ultralytics_plots")

## 6. Does the student agree with the teacher?
This compares the two on the **validation videos**, which the student never trained on.

In [ ]:
if RUN_DIR is not None:
    from broilerseg.segmentation import YoloSegmenter
    STUDENT_WEIGHTS = Path(RUN_DIR) / "weights" / "best.pt"
    student = YoloSegmenter(STUDENT_WEIGHTS, imgsz=IMGSZ)
else:
    from broilerseg.synthetic import ColorSegmenter
    STUDENT_WEIGHTS = None
    student = ColorSegmenter()
    student.name = "stand-in student (no training run)"
    print("No trained model: using a stand-in student so the plots can be shown.")

val_stems = ds[ds.split == "val"].stem.tolist() or ds.stem.tolist()
val_images = {s: load_frame(s) for s in val_stems}
agree, student_results = teacher_student_table(val_images, teacher_results, student)
agree.to_csv(OUT / "teacher_student_agreement.csv", index=False)
display(agree.groupby("class_name")[["iou"]].describe().round(3))
show(viz.plot_iou_distribution(agree), "iou_distribution")
show(viz.plot_count_agreement(agree), "count_agreement")

In [ ]:
per_img = agree.groupby("stem").iou.mean().sort_values()
for tag, ss in (("WORST", per_img.index[:3]), ("BEST", per_img.index[-2:])):
    for s in ss:
        show(viz.plot_teacher_student(val_images[s], teacher_results[s], student_results[s],
                                      f"{tag} agreement: {s} (mean IoU {per_img[s]:.2f})"),
             f"teacher_student_{tag.lower()}_{s[-10:]}")

## 7. Speed and size, and export for the Raspberry Pi
These timings come from **this** computer. On a Raspberry Pi 5, expect a nano model at 640 px with NCNN to run at roughly a few frames per second. Measure on the Pi itself. **NCNN** is the format Ultralytics recommends for ARM CPUs.

In [ ]:
bench_imgs = list(val_images.values())[:8]
rows = []
t_bench = benchmark(teacher, bench_imgs[:3], warmup=1)
teacher_mb = 0.0 if DRY_RUN else {"sam3": 3400, "grounded_sam2": 1800}[TEACHER]  # approx. download size
rows.append({**t_bench, "iou": 1.0, "size_mb": teacher_mb})
s_bench = benchmark(student, bench_imgs, warmup=2)
rows.append({**s_bench, "iou": float(agree.iou.mean()),
             "size_mb": path_size_mb(STUDENT_WEIGHTS) if STUDENT_WEIGHTS else 0.0})
EXPORTS = {}
if STUDENT_WEIGHTS is not None:
    from broilerseg.segmentation import YoloSegmenter
    EXPORTS = export_for_raspberry_pi(STUDENT_WEIGHTS, IMGSZ, EXPORT_FORMATS)
    for fmt, path in EXPORTS.items():
        seg = YoloSegmenter(path, imgsz=IMGSZ)
        b = benchmark(seg, bench_imgs, warmup=2)
        tab, _ = teacher_student_table(dict(list(val_images.items())[:8]), teacher_results, seg)
        rows.append({**b, "name": f"student {fmt}", "iou": float(tab.iou.mean()), "size_mb": path_size_mb(path)})
speed = pd.DataFrame(rows)
display(speed.round(3))
speed.to_csv(OUT / "speed.csv", index=False)
show(viz.plot_speed_accuracy(rows), "speed_accuracy")
for fmt, p in EXPORTS.items():
    print(f"{fmt}: {p}  ({path_size_mb(p):.1f} MB)  <- copy this to the Raspberry Pi")

## 8. Weight estimation with the student's masks
The real test is whether YOLO masks give weights as good as the teacher's masks. Paste your **FlockLabel export** (weight validation data, as in Notebook 1). Both segmenters then go through the same calibration, slicing and density steps.

In [ ]:
DENSITY = 900.0          # use the density chosen in Notebook 1 if you have it
SLICE_COUNT = None
MAX_FRAMES_PER_EVENT = 8
if DRY_RUN:
    WEIGHT_DIR = demo["export"]
else:
    WEIGHT_DIR = ask_for_directory("Paste the FlockLabel export folder (weight validation data)",
                                   env_var="BROILERSEG_WEIGHT_DIR")
export = load_flocklabel_export(WEIGHT_DIR)
wsamples = subsample_per_event(export.samples, MAX_FRAMES_PER_EVENT, SEED)
show(viz.plot_weight_dataset_overview(wsamples), "weight_data")
runs, exps = {}, {}
for name, seg in (("Teacher", teacher), ("Student", student)):
    e = WeightExperiment(seg, VolumeConfig(DENSITY, SLICE_COUNT))
    runs[name] = e.run(wsamples)
    exps[name] = e
    runs[name].to_csv(OUT / f"records_{name.lower()}.csv", index=False)
show(viz.plot_method_comparison({f"{n}: {s.name}": r for (n, r), s in zip(runs.items(), (teacher, student))},
                               "Weight accuracy: teacher masks vs student masks"), "weight_teacher_vs_student")
display(pd.DataFrame({n: regression_metrics(event_table(r).actual_kg, event_table(r).pred_kg)
                      for n, r in runs.items()}).T.round(3))

In [ ]:
for name, r in runs.items():
    show(viz.plot_per_event(r, f"{name}: every weighing event"), f"per_event_{name.lower()}")
both = runs["Teacher"][["sample_id", "pred_kg"]].merge(
    runs["Student"][["sample_id", "pred_kg"]], on="sample_id", suffixes=("_teacher", "_student"))
show(viz.plot_pred_vs_actual(both, pred="pred_kg_student", actual="pred_kg_teacher",
                             title="Frame by frame: student weight vs teacher weight",
                             xlabel="teacher-mask estimate (kg)", ylabel="student-mask estimate (kg)"), "student_vs_teacher_weight")
e = exps["Student"]
ok = runs["Student"].dropna(subset=["pred_kg"])
if len(ok):
    i = int(ok.sample_id.iloc[0])
    show(viz.plot_volume_storyboard(e.image(i), e.results[i], e.birds[i], e.estimates[i],
                                    ok.actual_kg.iloc[0], "Student pipeline, one frame"), "student_storyboard")

## 9. Deploying on the Raspberry Pi

```bash
# on the Pi (64-bit Raspberry Pi OS)
pip install ultralytics ncnn
pip install broilerseg            # or copy the package folder and: pip install -e .
```

```python
from broilerseg.segmentation import YoloSegmenter
from broilerseg.calibration import calibrate_from_mask
from broilerseg.segmentation import select_bird
from broilerseg.volume import estimate_volume

seg = YoloSegmenter("best_ncnn_model", imgsz=640)
res = seg.segment(frame_rgb)
cal = calibrate_from_mask(res.best("feeder").mask)
bird, _ = select_bird(res)                    # bird standing on the scale
kg = estimate_volume(bird.mask, cal.pixel_to_cm, density_kg_m3=DENSITY).mass_kg
```

**If the student is not good enough:** label more frames (especially those from section 3b), raise `FRAMES_PER_VIDEO`, try `yolov8s-seg.pt`, or train at the size you will deploy with.